In [ ]:
# =============================================================================
# XGBoost Ignition Model — 5×5 Spatial Aggregation
# Aggregates neighbouring pixels into 5×5 blocks before training.
# Each block takes:
#   - median of continuous features
#   - mode (most frequent) of land_cover
#   - max of target_ignition (fire if ANY pixel in block burned)
# Uses the same optimised hyperparameters as model_ignition_final.pkl
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
import glob
import os
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

os.makedirs('models',  exist_ok=True)
os.makedirs('figures', exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES         = DYNAMIC_FEATURES + STATIC_FEATURES
CONTINUOUS_FEATS = [f for f in FEATURES if f != 'land_cover']
CATEGORICAL_FEAT = 'land_cover'

TRAIN_YEARS    = list(range(2010, 2021))
VAL_YEARS      = [2021, 2022]
DOY_START_HARD = 91
DOY_END_HARD   = 310

BLOCK_SIZE     = 5      # 5×5 pixel aggregation window
AGG_METHOD     = 'median'   # 'median' or 'mean' for continuous features

# ── Load final model bundle ───────────────────────────────────────────────────
print("Loading final ignition model...")
bundle        = joblib.load('models/model_ignition_final.pkl')
CV_THRESHOLD  = bundle['cv_threshold']
best_params   = bundle['best_params']
print(f"  CV threshold : {CV_THRESHOLD:.4f}")
print(f"  Best params  : {best_params}")

# ── Load data year by year with memory-efficient approach ─────────────────────
KEEP_COLS = ['date', 'longitude', 'latitude'] + FEATURES + \
            ['target_ignition', 'fire_cause']

def load_and_filter(years, label):
    """Load CSVs year by year and apply seasonal filter immediately."""
    all_dfs = []
    for year in years:
        year_dfs = []
        for f in sorted(glob.glob(
                os.path.join(DATA_DIR, f'ML_{label}_{year}*.csv'))):
            try:
                df = pd.read_csv(f, usecols=KEEP_COLS,
                                 on_bad_lines='skip')
                year_dfs.append(df)
            except Exception as e:
                print(f"  WARNING: {os.path.basename(f)}: {e}")

        if not year_dfs:
            continue

        df_year = pd.concat(year_dfs, ignore_index=True)
        df_year['date_parsed'] = pd.to_datetime(
            df_year['date'], format='%Y%m%d', errors='coerce')
        df_year['doy'] = df_year['date_parsed'].dt.dayofyear
        df_year = df_year[
            (df_year['doy'] >= DOY_START_HARD) &
            (df_year['doy'] <= DOY_END_HARD)
        ].drop(columns=['date_parsed', 'doy'])

        df_year = df_year.dropna(
            subset=FEATURES + ['target_ignition', 'longitude', 'latitude'])
        df_year['target_ignition'] = df_year['target_ignition'].astype(int)
        df_year['land_cover']      = df_year['land_cover'].astype(int)
        df_year = df_year[df_year['target_ignition'].isin([0, 1])]

        n_fire = (df_year['target_ignition'] == 1).sum()
        print(f"  {year}: {len(df_year):>10,} rows  ({n_fire:,} fires)")
        all_dfs.append(df_year)

        import gc
        gc.collect()

    return pd.concat(all_dfs, ignore_index=True)

# ── Spatial aggregation function ──────────────────────────────────────────────
def assign_block_ids(df, block_size=BLOCK_SIZE):
    """
    Assign a block ID to each pixel based on its lon/lat position.
    Pixels within the same block_size × block_size neighbourhood
    and the same date share the same block_id.

    Strategy:
      1. Round lon/lat to a coarser grid by dividing by block spacing
         and taking the floor — this groups neighbouring pixels together.
      2. Combine with date to ensure temporal independence of blocks.

    The pixel spacing in GEE export is approximately 0.09 degrees
    at the default 9km GRID_SCALE. A 5×5 block therefore covers
    approximately 0.45 degrees in each direction.
    """
    # Estimate pixel spacing from the data
    pixel_spacing = df['longitude'].diff().abs()
    pixel_spacing = pixel_spacing[pixel_spacing > 0].median()
    block_spacing = pixel_spacing * block_size

    print(f"  Estimated pixel spacing : {pixel_spacing:.4f}°")
    print(f"  Block spacing (5×5)     : {block_spacing:.4f}°")

    # Assign block coordinates
    df = df.copy()
    df['block_lon'] = np.floor(df['longitude'] / block_spacing).astype(int)
    df['block_lat'] = np.floor(df['latitude']  / block_spacing).astype(int)
    df['block_id']  = (df['date'].astype(str) + '_' +
                       df['block_lon'].astype(str) + '_' +
                       df['block_lat'].astype(str))
    return df, block_spacing

def aggregate_blocks(df, method='median'):
    """
    Aggregate pixels within each block.
    - Continuous features: median or mean
    - land_cover: mode (most frequent class)
    - target_ignition: max (fire if ANY pixel in block burned)
    - date, block_lon, block_lat: first value (same within block)
    """
    print(f"  Aggregating {len(df):,} pixels into {df['block_id'].nunique():,} "
          f"blocks using {method}...")

    # Aggregation dictionary
    agg_dict = {}

    # Continuous features — median or mean
    for feat in CONTINUOUS_FEATS:
        agg_dict[feat] = method

    # Categorical feature — mode
    agg_dict[CATEGORICAL_FEAT] = lambda x: x.mode().iloc[0]

    # Target — max (fire if any pixel in block is fire)
    agg_dict['target_ignition'] = 'max'

    # Keep date and block coordinates
    agg_dict['date']      = 'first'
    agg_dict['block_lon'] = 'first'
    agg_dict['block_lat'] = 'first'

    df_agg = df.groupby('block_id').agg(agg_dict).reset_index()
    df_agg['land_cover'] = df_agg['land_cover'].astype(int)

    # Report class balance
    n_fire   = (df_agg['target_ignition'] == 1).sum()
    n_nofire = (df_agg['target_ignition'] == 0).sum()
    ratio    = n_nofire / max(n_fire, 1)
    print(f"  Aggregated: {len(df_agg):,} blocks  "
          f"({n_fire:,} fire + {n_nofire:,} no-fire  ratio {ratio:.0f}:1)")

    return df_agg

# ── Load and aggregate training data ──────────────────────────────────────────
print("\nLoading training data...")
df_train_raw = load_and_filter(TRAIN_YEARS, 'train')
print(f"  Total train rows (pixel level): {len(df_train_raw):,}")

print("\nAssigning block IDs to training data...")
df_train_raw, block_spacing = assign_block_ids(df_train_raw)

print("\nAggregating training blocks...")
df_train_agg = aggregate_blocks(df_train_raw, method=AGG_METHOD)

# ── Load and aggregate validation data ────────────────────────────────────────
print("\nLoading validation data...")
df_val_raw = load_and_filter(VAL_YEARS, 'val')
print(f"  Total val rows (pixel level): {len(df_val_raw):,}")

print("\nAssigning block IDs to validation data...")
df_val_raw, _ = assign_block_ids(df_val_raw)

print("\nAggregating validation blocks...")
df_val_agg = aggregate_blocks(df_val_raw, method=AGG_METHOD)

# ── Prepare arrays ─────────────────────────────────────────────────────────────
df_train_agg = df_train_agg.sort_values('date').reset_index(drop=True)
df_val_agg   = df_val_agg.sort_values('date').reset_index(drop=True)

X_train_agg  = df_train_agg[FEATURES].values
y_train_agg  = df_train_agg['target_ignition'].values
X_val_agg    = df_val_agg[FEATURES].values
y_val_agg    = df_val_agg['target_ignition'].values

n_pos    = y_train_agg.sum()
n_neg    = (y_train_agg == 0).sum()
ratio_agg = n_neg / max(n_pos, 1)

print(f"\nAggregated array shapes:")
print(f"  X_train_agg : {X_train_agg.shape}")
print(f"  X_val_agg   : {X_val_agg.shape}")
print(f"  Train ratio : {ratio_agg:.0f}:1")

# ── Compare pixel vs block level class ratios ──────────────────────────────────
print(f"\nClass ratio comparison:")
print(f"  Pixel level  : ~3,295:1  (original)")
print(f"  Block level  : {ratio_agg:.0f}:1  (after 5×5 aggregation)")
print(f"  Ratio reduced by factor of "
      f"{3295/ratio_agg:.1f}× due to max aggregation of target")

# ── Train aggregated model using same hyperparameters as final model ───────────
print("\nTraining aggregated ignition model...")
print("  Using same hyperparameters as model_ignition_final.pkl")

import time
t0 = time.time()

model_agg = xgb.XGBClassifier(
    objective             = 'binary:logistic',
    eval_metric           = 'auc',
    scale_pos_weight      = ratio_agg,    # recomputed for aggregated ratio
    early_stopping_rounds = 20,
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 1,
    **best_params
)

model_agg.fit(
    X_train_agg, y_train_agg,
    eval_set = [(X_val_agg, y_val_agg)],
    verbose  = 50
)

t1 = time.time()
print(f"\nTraining time : {(t1-t0)/60:.1f} minutes")
print(f"Best iteration: {model_agg.best_iteration}")

# ── Threshold CV for aggregated model ─────────────────────────────────────────
# Recompute threshold on aggregated training data
# The pixel-level CV threshold does not apply here because the
# probability distribution changes after spatial aggregation

print("\nComputing CV threshold for aggregated model...")

from sklearn.model_selection import StratifiedKFold
MIN_RECALL   = 0.70
N_FOLDS      = 5
candidate_thr = np.linspace(0.01, 0.99, 200)

skf_agg = StratifiedKFold(n_splits=N_FOLDS,
                           shuffle=True, random_state=42)
fold_thresholds_agg = []

for fold_idx, (tr_idx, vl_idx) in enumerate(
        skf_agg.split(X_train_agg, y_train_agg)):

    y_prob_fold = model_agg.predict_proba(X_train_agg[vl_idx])[:, 1]
    best_thr, best_prec = 0.5, 0.0

    for thr in candidate_thr:
        y_t  = (y_prob_fold >= thr).astype(int)
        rec  = recall_score(y_train_agg[vl_idx], y_t, zero_division=0)
        prec = precision_score(y_train_agg[vl_idx], y_t, zero_division=0)
        if rec >= MIN_RECALL and prec > best_prec:
            best_prec, best_thr = prec, thr

    fold_thresholds_agg.append(best_thr)
    print(f"  Fold {fold_idx+1}: threshold={best_thr:.3f}  "
          f"precision={best_prec:.4f}")

CV_THRESHOLD_AGG     = float(np.mean(fold_thresholds_agg))
CV_THRESHOLD_AGG_STD = float(np.std(fold_thresholds_agg))
print(f"\n  CV threshold (aggregated): "
      f"{CV_THRESHOLD_AGG:.4f} ± {CV_THRESHOLD_AGG_STD:.4f}")

# ── Evaluate aggregated model ──────────────────────────────────────────────────
print(f"\nEvaluating aggregated model on validation set...")

y_prob_agg = model_agg.predict_proba(X_val_agg)[:, 1]

# Evaluate at multiple thresholds
print(f"\n  {'Threshold':>10} {'Precision':>11} {'Recall':>9} "
      f"{'F1':>8} {'TP':>6} {'FP':>10} {'FN':>6}  FPR")
print(f"  {'-'*75}")

for thr in [0.5, CV_THRESHOLD_AGG, 0.3, 0.2, 0.1]:
    y_t = (y_prob_agg >= thr).astype(int)
    if len(np.unique(y_t)) < 2:
        continue
    tn, fp, fn, tp = confusion_matrix(y_val_agg, y_t).ravel()
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0
    rec  = tp/(tp+fn) if (tp+fn) > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
    fpr  = fp/(fp+tn)
    print(f"  {thr:>10.3f} {prec:>11.4f} {rec:>9.4f} "
          f"{f1:>8.4f} {tp:>6,} {fp:>10,} {fn:>6,}  {fpr:.4f}")

# Full evaluation at CV threshold
y_pred_agg = (y_prob_agg >= CV_THRESHOLD_AGG).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val_agg, y_pred_agg).ravel()

print(f"\nFull evaluation at CV threshold ({CV_THRESHOLD_AGG:.4f}):")
print(f"  ROC-AUC           : {roc_auc_score(y_val_agg, y_prob_agg):.4f}")
print(f"  Balanced accuracy : {balanced_accuracy_score(y_val_agg, y_pred_agg):.4f}")
print(classification_report(y_val_agg, y_pred_agg,
                            target_names=['No fire', 'Fire'],
                            digits=4, zero_division=0))
print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print(f"  Recall      : {tp/(tp+fn):.4f}  ({tp} of {tp+fn} blocks detected)")
print(f"  FPR         : {fp/(fp+tn):.4f}  "
      f"({fp/(fp+tn)*100:.1f}% no-fire blocks flagged)")

# ── Compare pixel vs block level performance ───────────────────────────────────
print(f"\n{'='*65}")
print("PIXEL vs BLOCK LEVEL PERFORMANCE COMPARISON")
print(f"{'='*65}")

# Load pixel level model for comparison
bundle_px    = joblib.load('models/model_ignition_final.pkl')
model_px     = bundle_px['model']
CV_THR_PX    = bundle_px['cv_threshold']

# Need pixel level val arrays — reload if not in memory
# X_val and y_ign_val should be in memory from the main training script
try:
    y_prob_px  = model_px.predict_proba(X_val)[:, 1]
    y_pred_px  = (y_prob_px >= CV_THR_PX).astype(int)
    tn_px, fp_px, fn_px, tp_px = confusion_matrix(
        y_ign_val, y_pred_px).ravel()

    print(f"\n  {'Metric':<30} {'Pixel (9km)':>14} {'Block (45km)':>14}")
    print(f"  {'-'*60}")
    print(f"  {'Spatial resolution':<30} {'~9 km':>14} {'~45 km':>14}")
    print(f"  {'Training observations':<30} "
          f"{len(X_train):>14,} {len(X_train_agg):>14,}")
    print(f"  {'Validation observations':<30} "
          f"{len(X_val):>14,} {len(X_val_agg):>14,}")
    print(f"  {'Class ratio':<30} "
          f"{'3,295:1':>14} {ratio_agg:.0f}:1{'>14'}")
    print(f"  {'ROC-AUC':<30} "
          f"{roc_auc_score(y_ign_val, y_prob_px):>14.4f} "
          f"{roc_auc_score(y_val_agg, y_prob_agg):>14.4f}")
    print(f"  {'CV threshold':<30} "
          f"{CV_THR_PX:>14.4f} {CV_THRESHOLD_AGG:>14.4f}")
    print(f"  {'Recall':<30} "
          f"{tp_px/(tp_px+fn_px):>14.4f} {tp/(tp+fn):>14.4f}")
    print(f"  {'False positive rate':<30} "
          f"{fp_px/(fp_px+tn_px):>14.4f} {fp/(fp+tn):>14.4f}")
    print(f"  {'True positives':<30} "
          f"{tp_px:>14,} {tp:>14,}")
    print(f"  {'False positives':<30} "
          f"{fp_px:>14,} {fp:>14,}")

except NameError:
    print("  Pixel level arrays not in memory — skipping comparison")
    print("  Run the main training script first to load X_val, y_ign_val")

# ── Save aggregated model ──────────────────────────────────────────────────────
joblib.dump({
    'model'              : model_agg,
    'cv_threshold'       : CV_THRESHOLD_AGG,
    'cv_threshold_std'   : CV_THRESHOLD_AGG_STD,
    'features'           : FEATURES,
    'best_params'        : best_params,
    'block_size'         : BLOCK_SIZE,
    'agg_method'         : AGG_METHOD,
    'block_spacing_deg'  : block_spacing,
    'train_ratio'        : ratio_agg,
    'train_years'        : TRAIN_YEARS,
    'val_years'          : VAL_YEARS,
}, 'models/model_ignition_agg5x5.pkl')

print(f"\nSaved: models/model_ignition_agg5x5.pkl")
print(f"\nSummary:")
print(f"  Block size         : {BLOCK_SIZE}×{BLOCK_SIZE} pixels")
print(f"  Spatial resolution : ~{block_spacing*111:.0f} km")
print(f"  Aggregation method : {AGG_METHOD} (continuous), "
      f"mode (land_cover), max (target)")
print(f"  CV threshold       : {CV_THRESHOLD_AGG:.4f} ± "
      f"{CV_THRESHOLD_AGG_STD:.4f}")